In [25]:
import torch

def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, unbiased=False, keepdim=True)
    x_hat = (x - mean) * torch.rsqrt(var + eps)
    return gamma * x_hat + beta

x = torch.randn(2, 4, 8)
gamma = torch.ones(8)
beta = torch.zeros(8)

ln = torch.nn.LayerNorm(8)
y = ln(x)
y_hat = layer_norm(x, gamma, beta)

print(torch.allclose(y, y_hat, atol=1e-6))                     # True
print((y - y_hat).abs().max())                                 # 最大误差，量级 ~1e-7

True
tensor(2.3842e-07, grad_fn=<MaxBackward1>)


In [21]:
x = torch.randn(2, 4, 8)
print(x.shape)                                # (2,4,8)
ln = torch.nn.LayerNorm(8)
print(ln.weight.shape, ln.bias.shape)         # (8,) (8,)  ← 只跟 normalized_shape 有关
print(ln(x).shape)                            # (2,4,8)   ← 形状保持

# 验证"只沿最后一维归一化"：
y = ln(x)
print(y.mean(-1).abs().max())                 # ≈ 0    每行均值为 0
print(y.std(-1, unbiased=False))              # ≈ 1    每行方差为 1（近似，受 eps 影响）

torch.Size([2, 4, 8])
torch.Size([8]) torch.Size([8])
torch.Size([2, 4, 8])
tensor(5.5879e-08, grad_fn=<MaxBackward1>)
tensor([[1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000]], grad_fn=<StdBackward0>)


In [8]:
import torch 
import torch.nn as nn

class MyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super(MyLayerNorm, self).__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = tuple(normalized_shape,)
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x):
        dims = tuple(range(-len(self.normalized_shape), 0))
        mean = x.mean(dim=dims, keepdim=True)
        var = x.var(dim=dims, unbiased=False, keepdim=True)
        x_hat = (x - mean) * torch.rsqrt(var + self.eps)
        return self.weight * x_hat + self.bias

In [13]:
torch.manual_seed(0)
x = torch.randn(2, 4, 8, dtype=torch.float64)

g0 = torch.randn(8, dtype=torch.float64)   # 随机 γ，让对比更有意义（别用全 1）
b0 = torch.randn(8, dtype=torch.float64)

mine = MyLayerNorm(8)
off  = nn.LayerNorm(8, eps=1e-5)
with torch.no_grad():
    mine.weight.copy_(g0); mine.bias.copy_(b0)
    off.weight.copy_(g0);  off.bias.copy_(b0)

y_mine = mine(x)
y_off  = off(x)

print("allclose :", torch.allclose(y_mine, y_off, atol=1e-12))
print("max err  :", (y_mine - y_off).abs().max().item())

RuntimeError: mixed dtype (CPU): all inputs must share same datatype.